# Double DQN Training - IMC Prosperity 4

**What this notebook does:**
1. Installs dependencies (stable-baselines3, gymnasium)
2. Uploads/mounts your trading data
3. Computes 19 trading indicators from order book data
4. Trains a Double DQN agent to trade EMERALDS & TOMATOES
5. Evaluates on held-out data and exports weights for submission

**Run this on:** Google Colab (free GPU) or Kaggle Notebooks

---

## 1. Setup & Install Dependencies

In [ ]:
!pip install stable-baselines3 gymnasium pandas numpy matplotlib --quiet
print("Dependencies installed!")

## 2. Upload Data

**Option A: Google Colab** - Upload from your local machine  
**Option B: Kaggle** - Add dataset to notebook  
**Option C: Google Drive** - Mount drive with repo cloned

Run ONE of the cells below:

In [ ]:
# === Google Colab Setup (DEFAULT) ===
# Clone your repo which already has the data and code
!git clone https://github.com/YOUR_USERNAME/EnM-Making-Waves---Prosperity-4.git /content/repo
DATA_DIR = "/content/repo/data"
PROJECT_ROOT = "/content/repo"

# === Alternative: Google Drive mount ===
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/EnM-Making-Waves---Prosperity-4/data"
# PROJECT_ROOT = "/content/drive/MyDrive/EnM-Making-Waves---Prosperity-4"

# === Alternative: Kaggle ===
# DATA_DIR = "/kaggle/input/prosperity4/data"
# PROJECT_ROOT = "/kaggle/input/prosperity4"

import sys
sys.path.insert(0, PROJECT_ROOT)
print(f"Data dir: {DATA_DIR}")
print(f"Project root: {PROJECT_ROOT}")

## 3. Load Data & Check GPU

In [ ]:
import torch
import numpy as np
import pandas as pd
import os

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected - training on CPU (still fast for this model)")

# Load data
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f"\nPrices: {len(prices_df)} rows")
print(f"Trades: {len(trades_df)} rows")
print(f"Products: {prices_df['product'].unique()}")
print(f"Days: {sorted(prices_df['day'].unique())}")

## 4. Compute Feature Normalization

Before training, we need to compute the mean and standard deviation of each feature across the training data. This ensures all 19 features are on the same scale.

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NETWORK_CONFIG, ENV_CONFIG
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

# Compute normalization parameters from training data
def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df["day"] == day) & (prices_df["product"] == product)]
            day_prices = day_prices.sort_values("timestamp").reset_index(drop=True)
            day_trades = trades_df[trades_df["symbol"] == product].sort_values("timestamp")
            for _, row in day_prices.iterrows():
                ts = row["timestamp"]
                ts_trades = day_trades[day_trades["timestamp"] == ts]
                trades = list(zip(ts_trades["price"], ts_trades["quantity"])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG["train_days"]
)

print("Feature normalization computed!")
print(f"Means (first 5): {feat_means[:5]}")
print(f"Stds  (first 5): {feat_stds[:5]}")

## 5. Create Environments & Train

This creates the training env (with data augmentation) and eval env (clean data), then trains the Double DQN agent.

**Adjust `TOTAL_TIMESTEPS` below** - more steps = better policy but longer training.

In [ ]:
# ============================================================
# TRAINING CONFIG - Adjust these!
# ============================================================
TOTAL_TIMESTEPS = 100_000   # Increase for better results (200k, 500k)
SEED = 42
LEARNING_RATE = 1e-3        # Set to None to use default from config
# ============================================================

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback

# Create training environment (with data augmentation)
train_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["train_days"][0],
    augment=True, seed=SEED,
)

# Create eval environment (no augmentation)
eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["eval_days"][0],
    augment=False, seed=SEED + 1,
)

# Set normalization params
for product in PRODUCTS:
    train_env.feature_computers[product].feature_means = feat_means
    train_env.feature_computers[product].feature_stds = feat_stds
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

# Output directory
MODEL_DIR = os.path.join(PROJECT_ROOT, "RLM", "double_dqn", "policy_weights")
os.makedirs(MODEL_DIR, exist_ok=True)

# Eval callback - saves best model during training
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=MODEL_DIR,
    eval_freq=max(TOTAL_TIMESTEPS // 20, 1000),
    n_eval_episodes=5,
    deterministic=True,
    verbose=1,
)

# Create DQN model (Double DQN is the SB3 default)
model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=LEARNING_RATE or DQN_CONFIG["learning_rate"],
    buffer_size=DQN_CONFIG["buffer_size"],
    learning_starts=DQN_CONFIG["learning_starts"],
    batch_size=DQN_CONFIG["batch_size"],
    gamma=DQN_CONFIG["gamma"],
    exploration_fraction=DQN_CONFIG["exploration_fraction"],
    exploration_initial_eps=DQN_CONFIG["exploration_initial_eps"],
    exploration_final_eps=DQN_CONFIG["exploration_final_eps"],
    target_update_interval=DQN_CONFIG["target_update_interval"],
    train_freq=DQN_CONFIG["train_freq"],
    gradient_steps=DQN_CONFIG["gradient_steps"],
    policy_kwargs=dict(net_arch=NETWORK_CONFIG["hidden_sizes"]),
    device="auto",  # Uses GPU if available
    verbose=1,
    seed=SEED,
)

print(f"Device: {model.device}")
print(f"Network: MLP {NETWORK_CONFIG['hidden_sizes']}")
print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")
print()

# TRAIN!
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=eval_callback,
    progress_bar=True,
)

# Save final model + normalization params
model.save(os.path.join(MODEL_DIR, "final_model"))
np.savez(os.path.join(MODEL_DIR, "norm_params.npz"),
         feature_means=feat_means, feature_stds=feat_stds)

print("\nTraining complete!")

## 6. Evaluate on Held-Out Day

Run the trained agent on day -1 (which it never saw during training) and check PnL.

In [ ]:
# Evaluate on held-out data
import matplotlib.pyplot as plt

n_eval_episodes = 10
episode_pnls = []
episode_rewards = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    total_reward = 0
    done = False
    pnl_curve = [0]

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        pnl_curve.append(info.get("pnl", total_reward))
        done = terminated or truncated

    episode_pnls.append(info.get("pnl", total_reward))
    episode_rewards.append(total_reward)
    print(f"Episode {ep+1}: PnL={episode_pnls[-1]:.2f}, Positions={info.get('positions', {})}")

pnls = np.array(episode_pnls)
print(f"\n{'='*40}")
print(f"Mean PnL:  {pnls.mean():.2f}")
print(f"Std PnL:   {pnls.std():.2f}")
print(f"Min PnL:   {pnls.min():.2f}")
print(f"Max PnL:   {pnls.max():.2f}")
if pnls.std() > 0:
    print(f"Sharpe:    {pnls.mean() / pnls.std():.2f}")

## 7. Export Weights to Numpy

Extract the trained PyTorch weights into a `.npz` file that can be used to build a pure-numpy submission.

In [ ]:
# Export trained weights to numpy .npz
weights = {}
layer_idx = 0
for name, param in model.q_net.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f"  {name}: {tensor.shape}")
    if "weight" in name:
        weights[f"W{layer_idx}"] = tensor
    elif "bias" in name:
        weights[f"B{layer_idx}"] = tensor
        layer_idx += 1

# Include normalization params
weights["feature_means"] = feat_means
weights["feature_stds"] = feat_stds

# Save
export_path = os.path.join(MODEL_DIR, "exported_weights.npz")
np.savez(export_path, **weights)

total_params = sum(v.size for k, v in weights.items() if "feature" not in k)
file_size = os.path.getsize(export_path)
print(f"\nExported {len(weights)} arrays, {total_params:,} parameters")
print(f"File size: {file_size / 1024:.1f} KB")
print(f"Saved to: {export_path}")

# Verify with numpy forward pass
from RLM.shared.numpy_policy import NumpyMLP
numpy_model = NumpyMLP(weights_path=export_path)
test_obs = np.random.randn(38).astype(np.float32)  # 19 features x 2 products
action, q_vals = numpy_model.predict(test_obs, normalize=False)
print(f"\nVerification pass: action={action}, Q-values={q_vals[:3]}...")

## 8. Download Weights

Run this cell to download the exported weights to your local machine. Then use `build_submission.py` locally to create the submission file.

In [ ]:
# Download weights (Google Colab only)
try:
    from google.colab import files
    files.download(export_path)
    print("Download started! Check your browser downloads.")
except ImportError:
    print("Not running on Colab. Weights saved to:", export_path)
    print("Download manually from the file browser on the left.")